# Discrete Time Markov Chains

In [ ]:
import glob
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

In [ ]:
files = sorted(glob.glob('../data/raw/**/*.csv', recursive=True))
print(f"Found {len(files)} files")

df = pd.concat([
    pd.read_csv(f, dtype={
        'start_station_id': 'str',
        'end_station_id':   'str',
        'member_casual':    'category',
        'rideable_type':    'category',
    })
    for f in files
], ignore_index=True)

df = df.dropna(subset=['start_station_id', 'end_station_id', 'started_at'])
df['started_at']  = pd.to_datetime(df['started_at'])
df['hour']        = df['started_at'].dt.hour
df['day_of_week'] = df['started_at'].dt.dayofweek  # 0=Mon, 6=Sun

print(f"Trips: {len(df):,}")
print(f"Date range: {df['started_at'].min()} → {df['started_at'].max()}")

In [ ]:
reference_date  = df['started_at'].max()
df['days_ago']  = (reference_date - df['started_at']).dt.days
decay           = 0.999
df['weight']    = decay ** df['days_ago']

print(f"Weight 1yr ago:  {decay**365:.3f}")
print(f"Weight 2yrs ago: {decay**730:.3f}")

In [ ]:
all_stations = pd.concat([df['start_station_id'], df['end_station_id']]).unique()
n_raw        = len(all_stations)
raw_to_idx   = {s: i for i, s in enumerate(all_stations)}

C_raw = np.zeros((n_raw, n_raw))
np.add.at(C_raw, (
    df['start_station_id'].map(raw_to_idx).values,
    df['end_station_id'].map(raw_to_idx).values
), df['weight'].values)

n_components, labels = connected_components(csr_matrix(C_raw), directed=True, connection='strong')
largest             = pd.Series(labels).value_counts().idxmax()
stations_to_keep    = [all_stations[i] for i, l in enumerate(labels) if l == largest]

df = df[df['start_station_id'].isin(stations_to_keep) &
        df['end_station_id'].isin(stations_to_keep)].copy()

all_stations    = sorted(pd.concat([df['start_station_id'], df['end_station_id']]).unique())
n               = len(all_stations)
station_to_idx  = {s: i for i, s in enumerate(all_stations)}
idx_to_station  = {i: s for s, i in station_to_idx.items()}

df['start_idx'] = df['start_station_id'].map(station_to_idx)
df['end_idx']   = df['end_station_id'].map(station_to_idx)

print(f"Stations kept: {n}, Trips: {len(df):,}")

In [ ]:
si = df['start_idx'].values
ei = df['end_idx'].values
dd = df['day_of_week'].values
hh = df['hour'].values
w  = df['weight'].values

C_global  = np.zeros((n, n))
C_per_day = np.zeros((7, n, n))
C_per_bin = np.zeros((7, 24, n, n))

np.add.at(C_global, (si, ei), w)

for day in range(7):
    mask = dd == day
    np.add.at(C_per_day[day], (si[mask], ei[mask]), w[mask])

for day in range(7):
    for hour in range(24):
        mask = (dd == day) & (hh == hour)
        np.add.at(C_per_bin[day, hour], (si[mask], ei[mask]), w[mask])
        
print("Count tensors built.")

In [ ]:
from scipy import linalg

alpha1 = 0.1
alpha2 = 0.01

def compute_pi(day, hour):
    C = C_per_bin[day, hour] + alpha1 * C_per_day[day] + alpha2 * C_global
    row_sums       = C.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    P              = C / row_sums
    eigenvalues, eigenvectors = linalg.eig(P.T)
    idx            = np.argmin(np.abs(eigenvalues - 1))
    pi             = eigenvectors[:, idx].real
    pi             = pi / pi.sum()
    return pi

print("Computing 168 stationary distributions...")
pi_by_day_hour = np.zeros((7, 24, n))
for day in range(7):
    for hour in range(24):
        pi_by_day_hour[day, hour] = compute_pi(day, hour)
    print(f"  day {day} done")
print("Done.")

In [ ]:
np.save('../outputs/pi_by_day_hour.npy', pi_by_day_hour)
print(f"Saved pi_by_day_hour.npy — shape {pi_by_day_hour.shape}")
# → (7, 24, n_stations)

In [ ]:
# average over days weighted by number of trips per day
pi_by_hour = np.zeros((24, n))
for hour in range(24):
    # average the 7 day distributions for this hour
    pi_by_hour[hour] = pi_by_day_hour[:, hour, :].mean(axis=0)
    pi_by_hour[hour] /= pi_by_hour[hour].sum()  # renormalize

np.save('../outputs/pi_by_hour.npy', pi_by_hour)
print(f"Saved pi_by_hour.npy — shape {pi_by_hour.shape}")

In [ ]:
arrivals   = C_global.sum(axis=0)
departures = C_global.sum(axis=1)
flow_ratio = arrivals / (departures + 1e-10)
flow_ratio /= np.median(flow_ratio)  # center around median

np.save('../outputs/flow_ratio.npy', flow_ratio)
print(f"Saved flow_ratio.npy — shape {flow_ratio.shape}")

In [ ]:
min_trips = 10
arrivals_by_hour   = np.zeros((24, n))
departures_by_hour = np.zeros((24, n))

for h in range(24):
    mask = hh == h
    np.add.at(arrivals_by_hour[h],   ei[mask], w[mask])
    np.add.at(departures_by_hour[h], si[mask], w[mask])

total_by_hour = arrivals_by_hour + departures_by_hour
flow_ratio_by_hour = np.where(
    total_by_hour >= min_trips,
    arrivals_by_hour / (departures_by_hour + 1e-10),
    1.0
)

np.save('../outputs/flow_ratio_by_hour.npy', flow_ratio_by_hour)
print(f"Saved flow_ratio_by_hour.npy — shape {flow_ratio_by_hour.shape}")

In [ ]:
# faster — precompute name/lat/lng lookups in one pass
import json

start_meta = df.groupby('start_station_id').agg(
    name=('start_station_name', 'first'),
    lat =('start_lat', 'median'),
    lng =('start_lng', 'median'),
)
end_meta = df.groupby('end_station_id').agg(
    name=('end_station_name', 'first'),
    lat =('end_lat', 'median'),
    lng =('end_lng', 'median'),
)

idx_to_info = {}
for idx, station_id in idx_to_station.items():
    if station_id in start_meta.index:
        row = start_meta.loc[station_id]
    else:
        row = end_meta.loc[station_id]
    idx_to_info[idx] = {
        'name': row['name'],
        'lat':  None if pd.isna(row['lat']) else float(row['lat']),
        'lng':  None if pd.isna(row['lng']) else float(row['lng']),
    }

with open('../outputs/stations.json', 'w') as f:
    json.dump(idx_to_info, f)

print(f"Saved stations.json — {len(idx_to_info)} stations")

In [ ]:
min_each = 5

arrivals_by_day_hour   = np.zeros((7, 24, n))
departures_by_day_hour = np.zeros((7, 24, n))

for d in range(7):
    for h in range(24):
        mask = (dd == d) & (hh == h)
        np.add.at(arrivals_by_day_hour[d, h],   ei[mask], w[mask])
        np.add.at(departures_by_day_hour[d, h], si[mask], w[mask])

flow_ratio_by_day_hour = np.where(
    (arrivals_by_day_hour >= min_each) & (departures_by_day_hour >= min_each),
    arrivals_by_day_hour / (departures_by_day_hour + 1e-10),
    1.0
)

np.save('../outputs/flow_ratio_by_day_hour.npy', flow_ratio_by_day_hour)
print(f"Saved flow_ratio_by_day_hour.npy — shape {flow_ratio_by_day_hour.shape}")

In [ ]:
print(f"pi_by_hour shape:        {np.load('../outputs/pi_by_hour.npy').shape}")
print(f"pi_by_day_hour shape:    {np.load('../outputs/pi_by_day_hour.npy').shape}")
print(f"flow_ratio shape:        {np.load('../outputs/flow_ratio.npy').shape}")
print(f"flow_ratio_by_hour:      {np.load('../outputs/flow_ratio_by_hour.npy').shape}")
print(f"flow_ratio_by_day_hour:  {np.load('../outputs/flow_ratio_by_day_hour.npy').shape}")

import json
with open('../outputs/stations.json') as f:
    s = json.load(f)
print(f"stations.json:           {len(s)} stations")